In [0]:
table_df = spark.table(
    "accelerator.metadata.table_configs"
)

In [0]:
pipelines = []

for row in table_df.collect():

    table_name = row["table_name"]
    pipelines.append({
    "pipeline_name": f"{table_name}_pipeline",
    "catalog" : "accelerator",
    "target": "gold",
    "serverless": "true",
    "libraries": [
        f"../src/bronze_{table_name}.py",
        f"../src/silver_{table_name}.py",
        f"../src/gold_{table_name}.py"
    ]
})
    

In [0]:
import json

print(
    json.dumps(
        pipelines,
        indent=4
    )
)

In [0]:

pipeline_yaml = ""

for pipeline in pipelines:

    libs = ""

    for file in pipeline["libraries"]:

      libs += f"""
              - file:
                  path: {file}
      """

    pipeline_yaml += f"""
    {pipeline['pipeline_name']}:

      name: {pipeline['pipeline_name']}
      catalog: {pipeline['catalog']}
      target: {pipeline['target']}
      development: true


      libraries:{libs}
"""

In [0]:
print(pipeline_yaml)

In [0]:
final_yaml = f"""
resources:
  pipelines:{pipeline_yaml}
"""

In [0]:
print(final_yaml)

In [0]:
pipeline_path = "/Volumes/accelerator/metadata/generated_code/bundle/resources/pipeline.yml"

with open(pipeline_path, "w") as f:
    f.write(final_yaml)

print("pipeline.yml generated")